![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--ai-memory-with-cognee--cognee-ai-memory)

# Building Intelligent AI Memory Systems with Cognee: A Python Development Knowledge Graph

This tutorial demonstrates how to construct an intelligent knowledge system that unifies authoritative Python practices, community standards, and personal development context into a coherent AI memory layer. By connecting contributions from Python's creator Guido van Rossum with established design principles and your own development patterns, you'll create a knowledge graph that produces contextually relevant, explainable, and consistent insights for Python development.


## Understanding the Architecture

Modern software development involves navigating multiple sources of truth: language documentation, community best practices, historical code examples, and personal experience. Traditional approaches treat these as separate resources, requiring manual synthesis and interpretation. This tutorial demonstrates a fundamentally different approach using knowledge graphs and retrieval-augmented generation.

The system we're building transforms scattered development data into an interconnected knowledge network. Rather than searching through isolated documents, you'll query a unified graph that understands relationships between Python's design philosophy, real-world implementation patterns from its creator, and your specific development context. This enables discovering non-obvious connections, such as how a type hinting challenge you faced relates to solutions Guido implemented in mypy, or how your async patterns align with Python's core design principles.

The power of this approach lies in its ability to infer implicit relationships and rules from your data. Through advanced graph algorithms and AI processing, the system identifies patterns that span across different data sources, creating a memory layer that grows more intelligent with each interaction.

## System Architecture and Data Flow


# <div align="center">
#   <img src="./data/cognee_diagram.png" alt="System Architecture and Data Flow" width="700"/>
# </div>



## Core Operations in Cognee

The Cognee framework provides four fundamental operations that transform raw data into intelligent knowledge systems. Each operation serves a specific purpose in the knowledge graph pipeline.

The **add()** function serves as the entry point for data ingestion, accepting various formats including JSON, Markdown, and API responses. This operation normalizes and prepares data for graph construction. The **cognify()** function represents the core transformation engine, using AI to extract entities, identify relationships, and structure data into a traversable knowledge graph. Through **search()**, users interact with the graph using natural language queries or structured Cypher expressions, with the system understanding context and relationships to return relevant results (Cypher queries are opt-in: set `ALLOW_CYPHER_QUERY=true` to enable them). Finally, **improve()** runs Cognee's self-improvement loop over the graph: it reviews dated facts for conflicts, builds a summary index that retrieval reads, and folds saved Q&A sessions and the feedback you give on them back into the permanent graph. (It replaces the older `memify()`, which is deprecated in Cognee 1.x.)

These operations work together to create a system that not only stores information but understands and connects it in meaningful ways.

## Data Sources and Their Roles

This tutorial leverages a carefully curated set of data sources that represent different perspectives on Python development. Each source contributes unique value to the knowledge graph.

The **guido_contributions.json** file contains actual pull requests and commits from Guido van Rossum's work on mypy and CPython. These provide authoritative examples of how Python's creator approaches language design and problem-solving. The **pep_style_guide.md** encodes community-accepted standards for Python code style and typing conventions, ensuring that generated insights align with established best practices. The **zen_principles.md** captures Python's philosophical foundation, grounding technical decisions in principles of simplicity, explicitness, and readability.

Personal context comes from **my_developer_rules.md**, which contains project-specific conventions and constraints, and **copilot_conversations.json**, which preserves actual development conversations including questions, code snippets, and discussion topics. This combination creates a comprehensive knowledge base that spans from language philosophy to practical implementation.

## Environment Setup and Configuration

Cognee's API is asynchronous, so every call is awaited. Jupyter supports top-level `await` in cells, so the notebook calls Cognee directly without any event-loop patching.

In [ ]:
%pip install "cognee==1.6.6"

## Model Configuration

For optimal balance between processing speed, cost efficiency, and output quality, this tutorial uses OpenAI's GPT-5.4 mini model. Ensure your environment configuration file (.env) contains the following setting:

```
LLM_MODEL="gpt-5.4-mini"
```

This model provides sufficient capability for entity extraction and relationship mapping while maintaining reasonable processing times for educational purposes.

### Verifying Cognee Installation

Confirm that Cognee is properly installed and identify whether you're using a local development version or an installed package.

In [ ]:
import cognee
import os
from pathlib import Path

print('Quick Cognee Import Check')
print('=' * 30)
print(f'Cognee location: {cognee.__file__}')
print(f'Package directory: {os.path.dirname(cognee.__file__)}')

# Check if it's local or installed
current_dir = Path.cwd()
cognee_path = Path(cognee.__file__)
if current_dir in cognee_path.parents:
    print('Status: LOCAL DEVELOPMENT VERSION')
else:
    print('Status: INSTALLED PACKAGE')

### Configuring Python Path

Ensure the Python interpreter can locate all necessary modules by adding the project root to the system path.

In [ ]:
import sys
from pathlib import Path
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
else:
    project_root = Path.cwd()

# Add project root to the beginning of sys.path
project_root_str = str(project_root.absolute())
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Project root: {project_root_str}")

### Initializing Clean State

Remove any existing Cognee data to ensure the tutorial starts with a fresh knowledge graph.

In [ ]:
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)

## Building the Knowledge Graph: Guido's Contributions

The knowledge graph construction begins with ingesting Guido van Rossum's development history. This dataset contains detailed pull requests and commits from his work on mypy and CPython, providing concrete examples of language design decisions and implementation patterns.

The ingestion process involves two key steps. First, the add() function loads the raw JSON data and assigns it to a named node set for organizational purposes. Then, cognify() processes this data to extract entities, identify relationships, and build temporal connections between different contributions. cognify() also extracts the dates it finds as timestamp nodes, which enables time-based analysis, allowing queries about the evolution of Python features over time.

### Loading and Processing Guido's Development Data

Ingest the contributions data and build the initial knowledge graph with temporal awareness.

In [ ]:
import cognee

result = await cognee.add(
    os.path.abspath("data/guido_contributions.json"),
    node_set=["guido_data"]
)
await cognee.cognify()

### Examining Initial Search Results

Display the first search result to verify successful graph construction.

In [ ]:
results = await cognee.search("Show me commits")
print(results[0])

## Understanding Graph Structure Through Visualization

The search operation demonstrates a fundamental difference from traditional database queries. Rather than retrieving isolated records, Cognee traverses a knowledge graph that understands relationships between commits, language features, design decisions, and their evolution over time. This enables discovery of patterns and connections that would be difficult to identify through conventional search methods.

Visualization provides crucial insights into the graph's structure and the relationships Cognee has identified. The interactive graph reveals clustering patterns around different projects and time periods, showing how ideas evolved into features and how different contributions relate to each other.

### Generating Interactive Graph Visualization

Create an HTML visualization of the knowledge graph structure.

In [ ]:
from cognee import visualize_graph
await visualize_graph('./guido_contributions.html')

### Displaying the Knowledge Graph

Render the interactive visualization within the notebook for exploration.

In [42]:
from IPython.display import IFrame, HTML, display
display(IFrame("./guido_contributions.html", width="100%", height="500"))

In [43]:
# Alternative method to open the graph directly in your default browser, if the IFrame doesn't work
import os
import webbrowser
from pathlib import Path

html_path = Path('guido_contributions.html').resolve()
file_url = html_path.as_uri()

print(f"HTML file path: {html_path}")
print(f"Opening: {file_url}")

webbrowser.open(file_url)


HTML file path: C:\Users\N7\PycharmProjects\agents-towards-production-private\tutorials\ai-memory-with-cognee\guido_contributions.html
Opening: file:///C:/Users/N7/PycharmProjects/agents-towards-production-private/tutorials/ai-memory-with-cognee/guido_contributions.html


True

## Graph Analysis and Pattern Recognition

The visualization reveals several important patterns in Python's development history. CPython core development shows concentrated activity around 2020, while mypy contributions focus specifically on fixtures and run classes. PEP discussions create connections to other contributors like Thomas Grainger and Adam Turner, showing the collaborative nature of language evolution. Time-based connections demonstrate how initial ideas and discussions evolved into concrete features and implementations.

These visual patterns help understand not just what changes were made, but how different aspects of Python development interconnect. The graph structure makes it possible to trace the evolution of ideas from initial proposals through implementation and refinement.

The interactive nature of the visualization allows for deeper exploration. You can examine specific clusters to understand focused development efforts or trace connections between seemingly unrelated contributions to discover hidden patterns in Python's evolution.

## Expanding the Knowledge Graph

With the foundation established through Guido's contributions, the next phase involves integrating additional data sources to create a comprehensive knowledge system. This expansion connects authoritative examples with community standards, design philosophy, and personal development context.

Each data source is assigned to a specific node set, creating logical groupings within the graph. Developer-specific data including conversations and personal rules forms one cluster, while principles and guidelines form another. This organization enables targeted searches within specific domains while maintaining connections across the entire graph.

### Ingesting Complete Data Set

Add all remaining data sources and process them into the unified knowledge graph.

In [ ]:
import os

await cognee.add(os.path.abspath("data/copilot_conversations.json"), node_set=["developer_data"])
await cognee.add(os.path.abspath("data/my_developer_rules.md"), node_set=["developer_data"])
await cognee.add(os.path.abspath("data/zen_principles.md"), node_set=["principles_data"])
await cognee.add(os.path.abspath("data/pep_style_guide.md"), node_set=["principles_data"])

await cognee.cognify()

### Discovering Cross-Domain Connections

Query the expanded graph to find connections between personal development challenges and Guido's solutions.

In [12]:
results = await cognee.search(
    "What validation issues did I encounter in January 2024, and how would they be addressed in Guido's contributions?",
    query_type=cognee.SearchType.GRAPH_COMPLETION
)
print(results)

[{'dataset_id': UUID('a86aa598-58c1-515d-965b-20594b7525b2'), 'dataset_name': 'main_dataset', 'dataset_tenant_id': None, 'search_result': ['In January 2024, the validation issue was that some mypyc tests still had untyped functions or missing annotations, which could let problems slip through.\n\nGuido’s contributions would address this by:\n- adding type annotations to test fixtures, and\n- enabling `check_untyped_defs = True` for `run-*` tests so unannotated functions are checked too.']}]


## Analyzing Connected Insights

The search results demonstrate how Cognee connects disparate data sources to provide contextual insights. The system identifies patterns between your development challenges and Guido's historical solutions, revealing connections such as circular import issues in type hints that mirror problems Guido solved in specific mypy pull requests, or performance optimizations in list comprehensions that follow patterns established in CPython commits.

These connections go beyond simple keyword matching. The graph understands the semantic relationships between different concepts, allowing it to recognize that a type hinting challenge you encountered relates conceptually to work done in mypy, even if the specific terminology differs.

## Advanced Memory Layer Construction

The improve() operation is Cognee's self-improvement loop. Once the graph is built, it runs a sequence of stages over it: it reviews dated facts for conflicting statements and records them, builds a retrieval-ready summary index of the graph, and, when you pass it session IDs, folds saved question and answer sessions and the feedback you gave on them back into the permanent graph. Unlike basic graph construction, this turns the graph into a memory layer that keeps learning from how you use it.

In this notebook we run improve() twice: once now over the documents alone, and once at the end to fold your feedback on a search result back into the graph. (Older Cognee versions called this step `memify()`; it is deprecated in Cognee 1.x.)

### Building the Intelligent Memory Layer

Run the self-improvement loop over the dataset and print what each stage did.

In [13]:
improve_result = await cognee.improve()
for stage in improve_result.stages:
    print(f"{stage.stage}: {stage.status} {stage.reason or ''}")

feedback_weights: skipped no_session_ids
persist_session_qa: skipped no_session_ids
persist_agent_traces: skipped no_session_ids
extract_agent_context: skipped no_session_ids
distill_sessions: skipped no_session_ids
update_user_preferences: skipped no_session_ids
build_truth_subspace: skipped no_session_ids
review_conflicts: completed 
global_context_index: completed 


## Memory Layer Capabilities

Stages that need session data (such as feedback weights, session distillation and user preferences) report `skipped` here because no session was passed yet; the document-level stages run. The summary index gives later searches a compact map of the whole graph, and the conflict review flags facts that contradict each other across sources.

Combined with the feedback step at the end of this notebook, this creates a feedback loop where the system learns from both authoritative sources and practical experience, continuously refining its understanding of effective Python development patterns.

### Analyzing Design Pattern Alignment

Query the memory-enhanced graph to understand how personal implementations align with Python philosophy.

In [14]:
# Search for connections between your async patterns and Python philosophy
results = await cognee.search(
    query_text= "How does my AsyncWebScraper implementation align with Python's design principles?",
    query_type=cognee.SearchType.GRAPH_COMPLETION
)
print("Python Pattern Analysis:", results)

Python Pattern Analysis: [{'dataset_id': UUID('a86aa598-58c1-515d-965b-20594b7525b2'), 'dataset_name': 'main_dataset', 'dataset_tenant_id': None, 'search_result': ['Your `AsyncWebScraper` aligns well with Python’s design principles:\n\n- **Explicit is better than implicit**: clear async context management, typed methods, and visible rate limiting.\n- **Simple is better than complex**: uses `asyncio`, `aiohttp`, and a semaphore instead of overengineering.\n- **Readability counts**: clean class structure, descriptive names, and straightforward control flow.\n- **Errors should never pass silently**: exceptions are caught and returned as structured errors.\n- **Practicality beats purity**: uses `TCPConnector`, timeouts, and concurrency limits for real-world scraping.\n- **There should be one obvious way to do it**: follows standard asyncio/aiohttp idioms.\n- **Context managers** support safe resource handling, which matches Python’s ethos.\n\nIn short: it’s practical, explicit, readable, a

## Targeted Search with Node Set Filtering

The node set organization established during data ingestion enables precise, targeted searches within specific domains of the knowledge graph. This filtering capability proves particularly valuable when seeking authoritative guidance on specific topics, as it allows queries to focus on relevant data sources while maintaining awareness of the broader context.

By constraining searches to particular node sets, you can ensure that responses draw from appropriate sources. Questions about style guidelines can be directed specifically to PEP documents and design principles, while implementation questions can focus on actual code examples and developer experiences.

### Searching Within Specific Knowledge Domains

Demonstrate targeted search by querying only the principles and guidelines node set.

In [48]:
from cognee.modules.engine.models.node_set import NodeSet
results = await cognee.search(
    query_text= "How should variables be named?",
    query_type=cognee.SearchType.GRAPH_COMPLETION,
    node_type=NodeSet,
    node_name=['principles_data']
)


2025-09-21T12:02:49.030834 [info     ] Graph projection completed: 5 nodes, 12 edges in 0.01s [CogneeGraph]

2025-09-21T12:02:49.962121 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.08s [cognee.shared.logging_utils]


## Temporal Analysis Capabilities

The timestamps cognify extracted during graph construction provide powerful capabilities for understanding how Python development has evolved over time. This temporal awareness allows queries that explore trends, identify periods of intense development activity, or understand how specific features emerged and matured.

Temporal queries can reveal insights about development velocity, the relationship between different features introduced in similar timeframes, or how implementation approaches have changed as the language has evolved. This historical perspective provides valuable context for understanding current best practices and anticipating future directions.

### Exploring Time-Based Development Patterns

Query the temporal aspects of the knowledge graph to understand development evolution.

In [ ]:
await cognee.search(
    query_text = "What did Guido contribute in 2025, and what can we learn from it?",
    query_type=cognee.SearchType.TEMPORAL
)

## Implementing Continuous Learning Through Feedback

The knowledge graph system supports continuous improvement through a feedback mechanism that captures the utility and relevance of search results. This creates a learning system that adapts to your specific needs and preferences over time.

Every search runs inside a session, and Cognee saves each question and answer to it. You can then attach a rating and a comment to any saved answer, and improve() folds that feedback into the graph, raising the weight of the graph elements behind answers you rated well. Searches use those weights when you pass `feedback_influence` (for example `feedback_influence=0.3`) or set `DEFAULT_FEEDBACK_INFLUENCE`; the default of 0 leaves ranking unchanged.

### Executing Search with Feedback Tracking

Perform a search inside a named session so the answer is saved and can receive feedback.

In [ ]:
answer = await cognee.search(
    query_type=cognee.SearchType.GRAPH_COMPLETION,
    query_text="What is the most zen thing about Python?",
    session_id="zen_session",  # The Q&A is saved to this session so it can receive feedback
)
print(answer)

### Providing Feedback to Enhance Future Searches

Rate the last answer in the session, then run improve() on that session so the feedback is applied to the graph.

In [18]:
# Fetch the last saved Q&A in the session and attach feedback to it
last_qa = (await cognee.session.get_session(session_id="zen_session", last_n=1))[-1]
await cognee.session.add_feedback(
    session_id="zen_session",
    qa_id=last_qa.qa_id,
    feedback_text="Last result was useful, I like code that complies with best practices.",
    feedback_score=5,  # 1 (poor) to 5 (great)
)

# Fold the session's Q&A and feedback back into the permanent graph
feedback_result = await cognee.improve(session_ids=["zen_session"])
for stage in feedback_result.stages:
    print(f"{stage.stage}: {stage.status} {stage.reason or ''}")

feedback_weights: completed 
persist_session_qa: completed 
persist_agent_traces: already_completed no_new_trace_steps
extract_agent_context: completed 
distill_sessions: completed 
update_user_preferences: skipped personalization_disabled
build_truth_subspace: skipped opt_in_disabled
review_conflicts: completed 
global_context_index: completed 


## Conclusion and Next Steps

This tutorial has demonstrated the construction of an intelligent knowledge system that unifies multiple perspectives on Python development into a coherent, queryable graph. The system you've built connects authoritative examples from Python's creator with community standards, design philosophy, and personal development experience, creating a powerful tool for informed decision-making in Python development.

The knowledge graph approach offers significant advantages over traditional documentation and search methods. By understanding relationships between different pieces of information, the system can provide insights that would be difficult or impossible to discover through conventional means. The temporal awareness, pattern recognition, and feedback mechanisms create a dynamic system that grows more valuable over time.

As you continue to use and expand this system, consider adding additional data sources such as project documentation, code reviews, or technical discussions. Each new data source enriches the graph, creating more connections and enabling more sophisticated insights. The feedback loop ensures that the system adapts to your specific needs, becoming an increasingly valuable partner in your Python development journey.

---
📖 **Want the decisions behind agents like this one, explained without code?** [*AI Agents Made Simple*](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=tutorials--ai-memory-with-cognee--cognee-ai-memory&click=book-nb-agents&target=https%3A%2F%2Fwww.amazon.com%2Fdp%2FB0HNXRSRH2&retarget=0&text=book-nb-agents) by the author of this repository covers this and much more. Available on Amazon.